In [107]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.model_selection import cross_val_score, StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge

plt.rcParams['lines.linewidth'] = 3
plt.rcParams['figure.figsize'] = [8, 5]
plt.rcParams['font.size'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 20
# plt.rcParams.keys()
from matplotlib import colors
from sklearn.linear_model import RidgeClassifier
from pygam import LogisticGAM, s, f

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [108]:
df = pd.read_csv('../../datasets/group_14.csv')
df["focus_factor"] = (
    df["focus_factor"]
    .astype(str)                  # ensure it's string
    .str.replace(",", ".", regex=False)  # replace comma with dot
)
df["focus_factor"] = pd.to_numeric(df["focus_factor"], errors="coerce").astype(float)

In [109]:
categorical_features = [
    'duration_1', 'duration_2', 'duration_3', 'duration_4', 'duration_5',
    'loudness_level',
    'popularity_level',
    'tempo_class',
    'explicit',
    'mode_indicator',
    'time_signature_class_boolean',
    'is_instrumental',
    'is_dance_hit',
    'echo_constant'
]

numerical_features = [
    'time_signature',
    'key_mode',
    'artist_song_count',
    'album_freq',
    'movement_index',
    'intensity_level',
    'verbal_density',
    'purity_score',
    'positivity_index',
    'activity_rate',
    'loudness_intensity',
    'happy_dance',
    'acoustics_instrumental',
    'artists_avg_popularity',
    'tempo_vs_genre',
    'energy_rank_pct',
    'loud_energy_ratio',
    'mood_pca',
    'mood_cluster',
    'acoustic_valence_mood_cluster',
    'signal_strength',
    'focus_factor',
    'ambient_level',
    'key_sin',
    'key_cos',
    'duration_log',
    'duration_log_z',
    'loudness_yeo',
    'temp_zscore',
    'resonance_factor',
    'timbre_index',
    'distorted_movement',
    'signal_power',
    'target_regression'
]


In [110]:
data_set = df.copy()
y = data_set['target_class']
X = data_set.drop(columns=["target_class"])

categorical_indices = [X.columns.get_loc(col) for col in categorical_features if col in X.columns]
numerical_indices = [X.columns.get_loc(col) for col in numerical_features if col in X.columns]



In [111]:

# 1. Scale your numerical features first
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled.iloc[:, numerical_indices] = scaler.fit_transform(X.iloc[:, numerical_indices])
X = X_scaled.values


In [112]:
# Build the terms
terms = s(numerical_indices[0])

for idx in numerical_indices[1:]:
    terms += s(idx)

for idx in categorical_indices:
    terms += f(idx)


In [113]:
# Encode the target to numeric
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Get number of classes
n_classes = len(np.unique(y_encoded))

# Train one LogisticGAM per class (one-vs-rest)
models = []
for class_idx in range(n_classes):
    print(f"\nTraining model for class {le.classes_[class_idx]}...")
    
    y_binary = (y_encoded == class_idx).astype(int)
    
    # Build terms
    terms = s(numerical_indices[0])
    for idx in numerical_indices[1:]:
        terms += s(idx)
    for idx in categorical_indices:
        terms += f(idx)
    
    # Create LogisticGAM with custom grid search parameters
    gam = LogisticGAM(terms)
    
    # Use coarser lambda grid to reduce search time and instability
    lam = np.logspace(-3, 3, 10)  # Fewer values than default
    
    # Suppress warnings during grid search
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        gam.gridsearch(X, y_binary, lam=lam)
    
    models.append(gam)
    print(f"Model {class_idx} completed!")

  0% (0 of 10) |                         | Elapsed Time: 0:00:00 ETA:  --:--:--



Training model for class class_45...


c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\links.py:148: RuntimeWarning: divide by zero encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\pygam.py:630: RuntimeWarning: invalid value encountered in multiply
  self.link.gradient(mu, self.distribution) ** 2
 10% (1 of 10) |##                       | Elapsed Time: 0:00:43 ETA:   0:06:28
 20% (2 of 10) |#####                    | Elapsed Time: 0:00:48 ETA:   0:03:14
c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\pygam.py:630: RuntimeWarning: overflow encountered in square
  self.link.gradient(mu, self.distribution) ** 2
 30% (3 of 10) |#######                  | Elapsed Time: 0:00:54 ETA:   0:02:07
c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\pygam.py:630: RuntimeWarning: overflow encountered in square
  self.link.gradient(mu, self.distribution) ** 2
 40

Model 0 completed!

Training model for class class_65...


 10% (1 of 10) |##                       | Elapsed Time: 0:00:26 ETA:   0:03:59
 20% (2 of 10) |#####                    | Elapsed Time: 0:00:31 ETA:   0:02:07
 30% (3 of 10) |#######                  | Elapsed Time: 0:00:37 ETA:   0:01:26
 40% (4 of 10) |##########               | Elapsed Time: 0:00:40 ETA:   0:01:00
 50% (5 of 10) |############             | Elapsed Time: 0:00:43 ETA:   0:00:43
 60% (6 of 10) |###############          | Elapsed Time: 0:00:46 ETA:   0:00:31
 70% (7 of 10) |#################        | Elapsed Time: 0:00:48 ETA:   0:00:20
 80% (8 of 10) |####################     | Elapsed Time: 0:00:50 ETA:   0:00:12
 90% (9 of 10) |######################   | Elapsed Time: 0:00:51 ETA:   0:00:05
100% (10 of 10) |########################| Elapsed Time: 0:00:53 Time:  0:00:53
  0% (0 of 10) |                         | Elapsed Time: 0:00:00 ETA:  --:--:--


Model 1 completed!

Training model for class class_73...


c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\links.py:148: RuntimeWarning: divide by zero encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\pygam.py:630: RuntimeWarning: invalid value encountered in multiply
  self.link.gradient(mu, self.distribution) ** 2
c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\pygam.py:630: RuntimeWarning: overflow encountered in square
  self.link.gradient(mu, self.distribution) ** 2
 10% (1 of 10) |##                       | Elapsed Time: 0:00:22 ETA:   0:03:26
c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\links.py:148: RuntimeWarning: divide by zero encountered in divide
  return dist.levels / (mu * (dist.levels - mu))
c:\Users\01\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygam\pygam.py:630: RuntimeWarning: overflow encountered in square
  self.link.gradient

Model 2 completed!
